# Lab 4 — Build the Audience, Then Buy It
Name: Maksym Dorofeiev | Group: KM-41

## Setup

In [12]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")

programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

## Task 1.0 — Clean the survey file

In [13]:
# Adult R/L panel members (aged 18+)
dw_rl = daily_weights[daily_weights["resp_type"].isin(["R", "L"])].copy()
dw_rl["bdate"] = pd.to_datetime(dw_rl["broadcast_date"]).dt.strftime("%Y-%m-%d")

ages = viewer_attributes[viewer_attributes["attr_id"] == 300].set_index("respondent_id")["value"].astype(float)
rl_ids = set(viewer_attributes[viewer_attributes["attr_id"].isin([101, 102])]["respondent_id"])
adult_ids = {r for r in rl_ids if ages.get(r, 0) >= 18}
child_ids = rl_ids - adult_ids

valid_kw = set(survey_layout[survey_layout["RECTYPE"] == "B"]["KEYWORD"])
yes_vals = {"1", "1.0", "y", "Y", "yes", "TRUE"}
pair_rank = {"X710101": 1, "X710102": 2, "X710103": 3}

df = survey.copy()
bad_log = []

# Repeated header / malformed trailer rows
is_hdr = df["RESPID"] == "RESPID"
bad_log.append(("Repeated header rows", int(is_hdr.sum()), "Dropped"))
df = df[~is_hdr].copy()

clean_id_str = df["RESPID"].str.strip().str.replace(r"\.0$", "", regex=True)
is_bad_id = ~clean_id_str.str.isdigit()
bad_log.append(("Non-numeric / trailer rows", int(is_bad_id.sum()), "Dropped"))
df = df[~is_bad_id].copy()

# RESPID formatting (whitespace, .0 suffix, leading zeros)
fmt_id = df["RESPID"].str.strip().str.replace(r"\.0$", "", regex=True).astype(int)
bad_id_fmt = (df["RESPID"] != fmt_id.astype(str)).sum()
bad_log.append(("Malformed RESPID (spaces, .0, leading 0)", int(bad_id_fmt), "Normalized to int"))
df["RESPID"] = fmt_id

# KEYWORD formatting (lowercase / trailing spaces)
norm_kw = df["KEYWORD"].str.strip().str.upper()
bad_kw_fmt = (df["KEYWORD"] != norm_kw).sum()
bad_log.append(("Malformed KEYWORD (case / spaces)", int(bad_kw_fmt), "Stripped and uppercased"))
df["KEYWORD"] = norm_kw

# VALUE issues (non-yes dropped, alt yes standardized)
is_yes = df["VALUE"].str.strip().isin(yes_vals)
bad_log.append(("Non-yes VALUE (0, 0.0, N, no, blank)", int((~is_yes).sum()), "Dropped"))
alt_yes = is_yes & (df["VALUE"] != "1")
bad_log.append(("Non-standard yes VALUE (1.0, y, Y, yes, TRUE)", int(alt_yes.sum()), "Standardized to '1'"))
df = df[is_yes].copy()
df["VALUE"] = "1"

# Out-of-layout keywords
bad_kw = ~df["KEYWORD"].isin(valid_kw)
bad_log.append(("Keyword not in layout", int(bad_kw.sum()), "Dropped"))
df = df[~bad_kw].copy()

# Under-18 panelists & non-panel / guest IDs
is_child = df["RESPID"].isin(child_ids)
bad_log.append(("Under-18 panel members", int(is_child.sum()), "Dropped"))
df = df[~is_child].copy()

is_unknown = ~df["RESPID"].isin(adult_ids)
bad_log.append(("Guests / non-panel IDs", int(is_unknown.sum()), "Dropped"))
df = df[~is_unknown].copy()

# Duplicate (RESPID, KEYWORD)
dupes = df.duplicated(["RESPID", "KEYWORD"])
bad_log.append(("Duplicate (RESPID, KEYWORD)", int(dupes.sum()), "Kept first"))
df = df[~dupes].copy()

# Conflicting pair counts (keep higher)
pair_rows = df[df["KEYWORD"].isin(pair_rank)].copy()
pair_rows["rank"] = pair_rows["KEYWORD"].map(pair_rank)
multi_pair_ids = pair_rows.groupby("RESPID").filter(lambda g: len(g) > 1)["RESPID"].unique()
drop_idx = pair_rows.sort_values("rank").duplicated("RESPID", keep="last")
drop_pair_idx = drop_idx[drop_idx].index
bad_log.append(("Conflicting pair counts (lower row)", int(len(drop_pair_idx)), "Kept higher pair count"))
df = df.drop(drop_pair_idx).copy()

# Missing 'Any' row for buyers with pair count or brand
has_any = set(df[df["KEYWORD"] == "X710100"]["RESPID"])
all_buyers = set(df["RESPID"])
missing_any = all_buyers - has_any
bad_log.append(("Missing X710100 (Any) row for buyer", len(missing_any), "Added X710100 row"))
if missing_any:
    add_df = pd.DataFrame({"RESPID": list(missing_any), "KEYWORD": "X710100", "VALUE": "1"})
    df = pd.concat([df, add_df], ignore_index=True)

# Build one-row-per-adult table
adults = pd.DataFrame({"respondent_id": sorted(adult_ids)})
kw_sets = {k: set(g["RESPID"]) for k, g in df.groupby("KEYWORD")}

adults["buyer"] = adults["respondent_id"].isin(kw_sets.get("X710100", set()))
adults["pairs"] = "0"
for code, label in [("X710101", "1"), ("X710102", "2"), ("X710103", "3+")]:
    adults.loc[adults["respondent_id"].isin(kw_sets.get(code, set())), "pairs"] = label
adults["stride"] = adults["respondent_id"].isin(kw_sets.get("X710201", set()))
adults["kestrel"] = adults["respondent_id"].isin(kw_sets.get("X710202", set()))
adults["easywalk"] = adults["respondent_id"].isin(kw_sets.get("X710203", set()))

# Raw vs clean check (for written answer)
raw_kestrel_rows = (survey["KEYWORD"] == "X710202").sum()
raw_kestrel_ids = survey[survey["KEYWORD"] == "X710202"]["RESPID"].nunique()
raw_buyer_ids = survey[survey["KEYWORD"] == "X710100"]["RESPID"].nunique()
clean_kestrel = int(adults["kestrel"].sum())
clean_buyer_share = adults["buyer"].mean()

print("Survey data rows:", len(survey))
print("Clean yes answers:", len(df))
print("Adult panel members:", len(adults))
print("Non-buyers:", int((~adults["buyer"]).sum()))
print("People with two pair counts:", len(multi_pair_ids))
print(f"Raw check -> Kestrel rows: {raw_kestrel_rows}, raw unique Kestrel IDs: {raw_kestrel_ids} (clean: {clean_kestrel})")
print(f"Raw check -> raw buyer share (X710100/8006): {raw_buyer_ids/len(adults):.3f} (clean: {clean_buyer_share:.3f})")

bad_table = pd.DataFrame(bad_log, columns=["Kind of bad record", "Rows", "Action"])
display(bad_table)

check_table = survey_layout[survey_layout["RECTYPE"] == "B"][["KEYWORD", "Text", "SAMPLE"]].copy()
check_table["SAMPLE"] = check_table["SAMPLE"].astype(int)
check_table["Your count"] = check_table["KEYWORD"].map(df["KEYWORD"].value_counts()).fillna(0).astype(int)
display(check_table[["KEYWORD", "Text", "Your count", "SAMPLE"]])

Survey data rows: 18349
Clean yes answers: 15635
Adult panel members: 8010
Non-buyers: 3231
People with two pair counts: 113
Raw check -> Kestrel rows: 516, raw unique Kestrel IDs: 495 (clean: 354)
Raw check -> raw buyer share (X710100/8006): 0.548 (clean: 0.597)


,Kind of bad record,Rows,Action
0,Repeated header rows,2,Dropped
1,Non-numeric / trailer rows,1,Dropped
2,"Malformed RESPID (spaces, .0, leading 0)",1455,Normalized to int
3,Malformed KEYWORD (case / spaces),1104,Stripped and uppercased
4,"Non-yes VALUE (0, 0.0, N, no, blank)",742,Dropped
5,"Non-standard yes VALUE (1.0, y, Y, yes, TRUE)",1194,Standardized to '1'
6,Keyword not in layout,738,Dropped
7,Under-18 panel members,273,Dropped
8,Guests / non-panel IDs,150,Dropped
9,"Duplicate (RESPID, KEYWORD)",933,Kept first


,KEYWORD,Text,Your count,SAMPLE
2,X710100,Any,4779,4779
3,X710101,1 pair,1810,1810
4,X710102,2 pairs,1308,1308
5,X710103,3 or more pairs,1661,1661
7,X710201,STRIDE,1563,1563
8,X710202,KESTREL,354,354
9,X710203,EASYWALK,1050,1050
10,X710298,OTHER BRAND,2739,2739
11,X710299,DON'T KNOW,371,371


**Answer:** the raw survey file violated every vendor guarantee: it contained repeated header rows, a non-numeric trailer row, malformed RESPID values (whitespace, .0 suffixes, and leading zeros), inconsistent KEYWORD casing/spaces, non-standard yes flags (1.0, y, Y, yes, TRUE) mixed with negative/blank rows (0, 0.0, N, no, ""), out-of-layout keywords, under-18 panelists, non-panel/guest IDs, duplicate (RESPID, KEYWORD) pairs, conflicting pair counts for the same person, and buyers missing the X710100 (Any) row. After cleaning, every keyword count matches the layout's SAMPLE column exactly (354 Kestrel buyers and 4,779 buyers out of 8,010 adult panelists, or 59.7%). Using the raw file as delivered would have overstated Kestrel buyers at 516 instead of 354 (off by 46%) and distorted the adult buyer share to 54.8% instead of 59.7%.

## Task 1.1 — Four definitions

In [14]:
adults["age"] = adults["respondent_id"].map(ages)

aud_sets = {
    "Adults 18+": set(adults["respondent_id"]),
    "Shoe buyers": set(adults[adults["buyer"]]["respondent_id"]),
    "Heavy buyers (3+)": set(adults[adults["pairs"] == "3+"]["respondent_id"]),
    "Kestrel buyers": set(adults[adults["kestrel"]]["respondent_id"]),
    "Adults 25-54": set(adults[adults["age"].between(25, 54)]["respondent_id"]),
}

# 31-day average daily in-tab weight sums
daily_univ = {
    name: dw_rl[dw_rl["respondent_id"].isin(ids)].groupby("bdate")["weight"].sum()
    for name, ids in aud_sets.items()
}
aud_size = {name: s.mean() for name, s in daily_univ.items()}

print(f"Adults 18+ (M): {aud_size['Adults 18+'] / 1e6:.1f}")
print(f"Shoe buyers (M): {aud_size['Shoe buyers'] / 1e6:.1f}")
print(f"Heavy buyers, 3+ pairs (M): {aud_size['Heavy buyers (3+)'] / 1e6:.1f}")
print(f"Kestrel buyers (M): {aud_size['Kestrel buyers'] / 1e6:.1f}")
print(f"Adults 25-54 (M): {aud_size['Adults 25-54'] / 1e6:.1f}")
print("Kestrel buyers in panel (n):", len(aud_sets["Kestrel buyers"]))

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


## Task 1.2 — Who are they?

In [15]:
male_ids = set(viewer_attributes[viewer_attributes["attr_id"] == 201]["respondent_id"])
female_ids = set(viewer_attributes[viewer_attributes["attr_id"] == 202]["respondent_id"])

adults["sex"] = np.where(adults["respondent_id"].isin(male_ids), "M", "W")
adults["age_grp"] = pd.cut(adults["age"], bins=[17, 34, 54, 150], labels=["18-34", "35-54", "55+"]).astype(str)
adults["demo_grp"] = adults["sex"] + adults["age_grp"]

groups = ["M18-34", "W18-34", "M35-54", "W35-54", "M55+", "W55+"]
grp_map = adults.set_index("respondent_id")["demo_grp"]

dw_adults = dw_rl[dw_rl["respondent_id"].isin(aud_sets["Adults 18+"])].copy()
dw_adults["demo_grp"] = dw_adults["respondent_id"].map(grp_map)

# Shares per audience using 31-day average daily sums
share_df = pd.DataFrame(index=list(aud_sets.keys()), columns=groups, dtype=float)
for name, ids in aud_sets.items():
    sub = dw_adults[dw_adults["respondent_id"].isin(ids)]
    grp_avg = sub.groupby(["bdate", "demo_grp"])["weight"].sum().unstack(fill_value=0).mean()
    for g in groups:
        share_df.loc[name, g] = grp_avg.get(g, 0.0) / aud_size[name]

index_df = share_df.div(share_df.loc["Adults 18+"], axis=1) * 100

# Combined display table
profile_rows = []
for name in aud_sets:
    row = {"Audience": name}
    for g in groups:
        row[f"{g} share"] = round(share_df.loc[name, g], 3)
        row[f"{g} idx"] = round(index_df.loc[name, g], 1)
    profile_rows.append(row)

display(pd.DataFrame(profile_rows).set_index("Audience"))

,M18-34 share,M18-34 idx,W18-34 share,W18-34 idx,M35-54 share,M35-54 idx,W35-54 share,W35-54 idx,M55+ share,M55+ idx,W55+ share,W55+ idx
Audience,,,,,,,,,,,,
Adults 18+,0.144,100.0,0.144,100.0,0.160,100.0,0.160,100.0,0.185,100.0,0.208,100.0
Shoe buyers,0.140,97.4,0.140,97.6,0.158,99.2,0.178,111.4,0.186,100.8,0.197,94.6
Heavy buyers (3+),0.111,77.1,0.139,97.0,0.138,86.8,0.226,141.9,0.189,102.2,0.196,94.0
Kestrel buyers,0.118,82.1,0.181,126.0,0.189,118.6,0.265,165.8,0.126,68.2,0.121,58.0
Adults 25-54,0.174,121.2,0.174,121.2,0.326,204.1,0.326,204.1,0.000,0.0,0.000,0.0


**Answer:** I would hand the planner heavy athletic-shoe buyers (3+ pairs) because it directly matches Kestrel's brief, sharpens demographic targeting toward women 35-54 (index 142) and away from men 18-34 (index 77), and still has a large enough panel base (1,661 panelists) to produce reliable program ratings, unlike Kestrel buyers (only 354 panelists), which is too small for stable TV planning. "Athletic-shoe buyers" looks almost identical to all adults 18+ (indices close to 100 across all six age/sex groups) because roughly 60% of all adults bought at least one pair in the last 12 months, so the segment is too broad to separate from the general population. Because the purchase data was fused onto the panel using only age, sex, and household income, it cannot tell you about true behavioral or interest-based program affinity within those demographics, for example, whether dedicated runners watch more live sports or fitness programming than non-runners of the same age, sex, and income

## Task 2.1 — Programs by audience

In [16]:
# Prepare R/L viewings grouped by network for fast overlap calculation
v_rl = viewings[viewings["resp_type"].isin(["R", "L"])][
    ["respondent_id", "network_code", "start_ts", "end_ts", "weight"]
].copy()
v_by_net = {net: g for net, g in v_rl.groupby("network_code")}

# Helper: average-minute viewers and rating per telecast for given audiences
def rate_telecasts(telecasts_df, target_aud_dict, daily_univ_dict):
    rows = []
    for _, tc in telecasts_df.iterrows():
        net = tc["network_code"]
        bdate = pd.to_datetime(tc["broadcast_date"]).strftime("%Y-%m-%d")
        t_start, t_end, dur = tc["start_ts"], tc["end_ts"], tc["duration_min"]

        v = v_by_net.get(net)
        if v is not None:
            sub = v[(v["start_ts"] < t_end) & (v["end_ts"] > t_start)]
            ov = (sub["end_ts"].clip(upper=t_end) - sub["start_ts"].clip(lower=t_start)).dt.total_seconds() / 60.0
            w_min = sub["weight"] * ov
            rids = sub["respondent_id"]
        else:
            w_min = pd.Series(dtype=float)
            rids = pd.Series(dtype=int)

        rec = {"telecast_id": tc["telecast_id"]}
        for name, ids in target_aud_dict.items():
            mask = rids.isin(ids) if len(rids) else []
            viewers = w_min[mask].sum() / dur if len(rids) else 0.0
            univ = daily_univ_dict[name].get(bdate, np.nan)
            rec[f"viewers_{name}"] = viewers
            rec[f"rating_{name}"] = viewers / univ * 100.0
        rows.append(rec)
    return pd.DataFrame(rows)

# Helper: price each telecast's :30 from rate_card (Lab 2 rule: program rows win over time rows)
def to_bcast_min(t_str):
    h, m = map(int, str(t_str).split(":"))
    return h * 60 + m - 360

def price_telecasts(telecasts_df):
    df = telecasts_df[["telecast_id", "network_code", "program_name", "start_ts"]].copy()
    rc_prog = rate_card[rate_card["program_name"].fillna("") != ""][["network_code", "program_name", "cost_30s_usd"]]
    df = df.merge(rc_prog, on=["network_code", "program_name"], how="left")

    bts = pd.to_datetime(df["start_ts"]) - pd.Timedelta(hours=6)
    df["bday"] = bts.dt.weekday
    df["bmin"] = bts.dt.hour * 60 + bts.dt.minute

    rc_time = rate_card[rate_card["program_name"].fillna("") == ""].copy()
    rc_time["start_min"] = rc_time["start_time"].apply(to_bcast_min)
    rc_time["end_min"] = rc_time["end_time"].apply(to_bcast_min)
    rc_time = rc_time.explode("weekdays").rename(columns={"weekdays": "bday"})
    rc_time["bday"] = rc_time["bday"].astype(int)

    miss = df[df["cost_30s_usd"].isna()][["telecast_id", "network_code", "bday", "bmin"]].merge(
        rc_time[["network_code", "bday", "cost_30s_usd", "start_min", "end_min"]],
        on=["network_code", "bday"], how="left"
    )
    miss = miss[(miss["bmin"] >= miss["start_min"]) & (miss["bmin"] < miss["end_min"])].drop_duplicates("telecast_id")
    df["cost_30s_usd"] = df["cost_30s_usd"].fillna(df["telecast_id"].map(miss.set_index("telecast_id")["cost_30s_usd"]))
    return df.set_index("telecast_id")["cost_30s_usd"]

# Eight Prime programs
t2_progs = [
    ("NBC", "NFL Football"),
    ("ABC", "NFL Football"),
    ("ABC", "Dancing with the Stars"),
    ("CBS", "NCIS"),
    ("FNC", "Hannity"),
    ("HGTV", "Unsellable Houses"),
    ("FOOD", "Halloween Wars"),
    ("FX", "Grotesquerie"),
]

prime_t2 = programs[(programs["daypart"] == "Prime") & (programs.set_index(["network_code", "program_name"]).index.isin(t2_progs))].copy()
t2_auds = {k: aud_sets[k] for k in ["Adults 18+", "Heavy buyers (3+)", "Kestrel buyers"]}
t2_rates = rate_telecasts(prime_t2, t2_auds, daily_univ)
prime_t2 = prime_t2.merge(t2_rates, on="telecast_id")
prime_t2["cost_30s"] = prime_t2["telecast_id"].map(price_telecasts(prime_t2))

# Aggregate by program
rows = []
for net, pname in t2_progs:
    g = prime_t2[(prime_t2["network_code"] == net) & (prime_t2["program_name"] == pname)]
    a_rat = g["rating_Adults 18+"].mean()
    t_rat = g["rating_Heavy buyers (3+)"].mean()
    k_rat = g["rating_Kestrel buyers"].mean()
    a_vw = g["viewers_Adults 18+"].mean()
    t_vw = g["viewers_Heavy buyers (3+)"].mean()
    cost = g["cost_30s"].mean()
    rows.append({
        "Program": f"{pname} ({net})",
        "Telecasts": len(g),
        "Adults rating": round(a_rat, 2),
        "Target rating": round(t_rat, 2),
        "Target viewers (K)": round(t_vw / 1e3, 1),
        "Composition": round(t_vw / a_vw, 3),
        "Index": round(t_rat / a_rat * 100, 1),
        "Cost :30 ($)": round(cost),
        "Adults CPM ($)": round(cost / a_vw * 1000),
        "Target CPM ($)": round(cost / t_vw * 1000),
        "Kestrel index": round(k_rat / a_rat * 100, 1),
    })

t2_df = pd.DataFrame(rows)
display(t2_df.drop(columns=["Telecasts", "Kestrel index"]))
print(f"Target universe (M): {aud_size['Heavy buyers (3+)'] / 1e6:.1f}")

,Program,Adults rating,Target rating,Target viewers (K),Composition,Index,Cost :30 ($),Adults CPM ($),Target CPM ($)
0,NFL Football (NBC),5.82,5.79,2865.0,0.195,99.3,487200,33,170
1,NFL Football (ABC),8.93,8.36,4149.1,0.184,93.6,723600,32,174
2,Dancing with the Stars (ABC),3.25,3.14,1564.2,0.191,96.8,158600,19,101
3,NCIS (CBS),2.21,2.43,1206.3,0.216,110.1,178800,32,148
4,Hannity (FNC),1.09,0.84,416.9,0.152,77.1,39900,15,96
5,Unsellable Houses (HGTV),0.50,0.64,317.2,0.249,126.8,12900,10,41
6,Halloween Wars (FOOD),0.28,0.33,161.9,0.228,116.2,9700,14,60
7,Grotesquerie (FX),0.26,0.25,124.2,0.188,95.9,7592,11,61


Target universe (M): 49.6


## Task 2.2 — Index, volume or price?

In [17]:
print("Highest index (program):", t2_df.loc[t2_df["Index"].idxmax(), "Program"])
print("Most target viewers (program):", t2_df.loc[t2_df["Target viewers (K)"].idxmax(), "Program"])
print("Lowest target CPM (program):", t2_df.loc[t2_df["Target CPM ($)"].idxmin(), "Program"])

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): NFL Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer:** they are not all the same program: Unsellable Houses (HGTV) has the highest index, NFL Football (ABC) (Monday Night Football) delivers the most heavy-buyer viewers, and Unsellable Houses (HGTV) is again the cheapest per target viewer. I would tell a client demanding "only the highest index" that index measures audience concentration, not scale or price (just like Lecture 5's shampoo example). High-index shows are often smaller cable programs with modest reach, so buying only the top-indexing title caps total delivery and misses most heavy buyers, whereas balancing target CPM and reach volume delivers far more target impressions for the same budget

## Task 2.3 — Kestrel buyers

In [18]:
display(t2_df[["Program", "Index", "Kestrel index"]].rename(columns={"Index": "Heavy-buyer index"}))

,Program,Heavy-buyer index,Kestrel index
0,NFL Football (NBC),99.3,98.5
1,NFL Football (ABC),93.6,85.9
2,Dancing with the Stars (ABC),96.8,82.7
3,NCIS (CBS),110.1,56.5
4,Hannity (FNC),77.1,42.6
5,Unsellable Houses (HGTV),126.8,76.9
6,Halloween Wars (FOOD),116.2,95.2
7,Grotesquerie (FX),95.9,129.6


**Answer:** the Kestrel-buyer index jumps around far more than the heavy-buyer index because of small-sample noise: there are only 354 Kestrel buyers in the entire panel (~4.4% of adults) versus 1,661 heavy buyers. On an individual show—especially lower-rated cable telecasts—only a handful of Kestrel panelists tune in, so one or two extra viewers cause huge swings in the rating and index. I would trust the heavy-buyer index because its ~4.7x larger panel base yields much lower sampling error. To check reliability, you can inspect the unweighted count of in-target panelists watching each program, compute binomial standard errors / confidence intervals around the ratings, or split October into two halves to see whether the index stays consistent

## Task 3.1 — Weights

In [19]:
# Lab 3 ACR rules: Eastern time, 1-min edit rule, match to exactly 1 household
acr = acr_sessions.copy()
for col in ["start_utc", "end_utc"]:
    s = acr[col]
    acr[col] = (s.dt.tz_localize("UTC") if s.dt.tz is None else s).dt.tz_convert("America/New_York")

acr["bcast_date"] = (acr["start_utc"] - pd.Timedelta(hours=6)).dt.strftime("%Y-%m-%d")
acr["duration_min"] = (acr["end_utc"] - acr["start_utc"]).dt.total_seconds() / 60.0
acr_clean = acr[acr["duration_min"] >= 1.0].copy()

ip_cnt = hh_attributes.groupby("hashed_ip")["household_id"].nunique()
valid_ips = set(ip_cnt[ip_cnt == 1].index)
hh_one = hh_attributes[hh_attributes["hashed_ip"].isin(valid_ips)][["hashed_ip", "head_age_band", "income_band"]].copy()
hh_one["head_age"] = np.where(hh_one["head_age_band"] == "65+", "65+", "Under 65")

tv_hh = acr[["tv_id", "hashed_ip"]].drop_duplicates().merge(hh_one, on="hashed_ip", how="inner")
acr_m = acr_clean.merge(tv_hh[["tv_id", "head_age", "income_band"]], on="tv_id", how="inner")

# Daily cell weights = universe tv_households / matched in-tab TVs in cell that day
intab_cells = acr_m.groupby(["bcast_date", "head_age", "income_band"])["tv_id"].nunique().reset_index(name="n_tvs")
cell_w = intab_cells.merge(universe, on=["head_age", "income_band"], how="left")
cell_w["tv_weight"] = cell_w["tv_households"] / cell_w["n_tvs"]

# Verify daily sum of weights
daily_w_sum = (cell_w["tv_weight"] * cell_w["n_tvs"]).groupby(cell_w["bcast_date"]).sum()
daily_intab_m = acr_m.groupby("bcast_date")["tv_id"].nunique()

# Ratings on reference_ratings telecasts
progs_et = programs.copy()
for col in ["start_ts", "end_ts"]:
    s = progs_et[col]
    progs_et[col] = s.dt.tz_localize("America/New_York") if s.dt.tz is None else s.dt.tz_convert("America/New_York")
progs_et["bcast_date"] = pd.to_datetime(progs_et["broadcast_date"]).dt.strftime("%Y-%m-%d")

ref_p = reference_ratings.merge(progs_et, on="telecast_id", how="left")
acr_live_m = acr_m[acr_m["session_type"] == "live"].merge(
    cell_w[["bcast_date", "head_age", "income_band", "tv_weight"]],
    on=["bcast_date", "head_age", "income_band"], how="left"
)

t3_rows = []
for _, r in ref_p.iterrows():
    bdate, net = r["bcast_date"], r["network_code"]
    sub = acr_live_m[(acr_live_m["bcast_date"] == bdate) & (acr_live_m["network_code"] == net)]
    ov = (sub["end_utc"].clip(upper=r["end_ts"]) - sub["start_utc"].clip(lower=r["start_ts"])).dt.total_seconds() / 60.0
    watched = sub[ov >= 1.0].drop_duplicates("tv_id")

    unw = len(watched) / daily_intab_m.loc[bdate] * 100.0
    wtd = watched["tv_weight"].sum() / daily_w_sum.loc[bdate] * 100.0
    ref_r = r["hh_rating"]
    t3_rows.append({
        "Telecast": f"{r['program_name']} ({net})",
        "Date": bdate,
        "Unweighted": round(unw, 2),
        "Weighted": round(wtd, 2),
        "Reference": round(ref_r, 2),
    })

t3_df = pd.DataFrame(t3_rows)
gap_unw = (t3_df["Reference"] - t3_df["Unweighted"]).abs().mean()
gap_wtd = (t3_df["Reference"] - t3_df["Weighted"]).abs().mean()
share_closed = (gap_unw - gap_wtd) / gap_unw

print("TVs matched to one household:", tv_hh["tv_id"].nunique())
print(f"Weights sum per day (M): {daily_w_sum.mean() / 1e6:.1f}")
display(t3_df)
print(f"Mean gap unweighted (points): {gap_unw:.2f}")
print(f"Mean gap weighted (points): {gap_wtd:.2f}")
print(f"Share of gap closed: {share_closed:.3f}")

TVs matched to one household: 19607
Weights sum per day (M): 125.8


,Telecast,Date,Unweighted,Weighted,Reference
0,MLB Postseason (FOX),2024-10-25,17.03,17.76,25.86
1,NFL Football (NBC),2024-10-27,22.62,23.75,33.14
2,NCIS (CBS),2024-10-14,5.50,5.80,10.56
3,Hannity (FNC),2024-10-16,1.04,1.35,2.30
4,Unsellable Houses (HGTV),2024-10-16,2.12,2.08,3.69
5,Halloween Baking Championship (FOOD),2024-10-14,0.95,0.90,1.63
6,Grotesquerie (FX),2024-10-16,2.17,2.11,3.53
7,The Real Housewives of New York City (BRAVO),2024-10-15,0.71,0.64,1.23


Mean gap unweighted (points): 3.72
Mean gap weighted (points): 3.44
Share of gap closed: 0.076


## Task 3.2 — What weighting cannot fix

**Answer:** weighting moved older-skewing programs - especially Hannity (FNC) and NCIS (CBS) - upward toward the reference ratings, while younger-skewing shows like Grotesquerie (FX) and The Real Housewives of New York City (BRAVO) barely moved or edged slightly down, because weighting corrects the smart-TV footprint's under-representation of 65+ households. Even after weighting, most of the gap to the reference remains (only 7.6% closed) because demographic weights cannot fix single-TV-per-home under-coverage (missing viewing on secondary sets in multi-TV homes), unrecognized/unknown ACR sessions and external streaming devices, within-cell opt-in/OEM selection bias, or unmatched/shared-IP households (e.g., apartments). Weighting by income barely moves these ratings because linear TV viewing of these mainstream networks is driven primarily by age (older homes watch much more broadcast and cable news), whereas viewing levels vary very little across income bands once age is controlled for

## Task 4.1 — October predicts October

In [20]:
big5 = ["CBS", "NBC", "ABC", "FOX", "FNC"]
p_big5 = programs[(programs["daypart"] == "Prime") & (programs["network_code"].isin(big5))].copy()
p_big5["bdate"] = pd.to_datetime(p_big5["broadcast_date"]).dt.strftime("%Y-%m-%d")

hist = p_big5[(p_big5["bdate"] >= "2024-10-01") & (p_big5["bdate"] <= "2024-10-21")].copy()
fc = p_big5[(p_big5["bdate"] >= "2024-10-22") & (p_big5["bdate"] <= "2024-10-28")].copy()

# Keep forecast telecasts whose (network, program) aired in history
hist_keys = set(zip(hist["network_code"], hist["program_name"]))
fc = fc[fc.set_index(["network_code", "program_name"]).index.isin(hist_keys)].copy()

# Add P2+ daily universe
p2_ids = set(dw_rl["respondent_id"].unique())
t4_auds = {"P2+": p2_ids, "Heavy": aud_sets["Heavy buyers (3+)"]}
t4_univ = {"P2+": dw_rl.groupby("bdate")["weight"].sum(), "Heavy": daily_univ["Heavy buyers (3+)"]}

# Compute actual ratings for history & forecast week
all_t4 = pd.concat([hist, fc], ignore_index=True)
t4_rates = rate_telecasts(all_t4, t4_auds, t4_univ)
hist = hist.merge(t4_rates, on="telecast_id")
fc = fc.merge(t4_rates, on="telecast_id")

# Program-level mean history rating
hist_mean = hist.groupby(["network_code", "program_name"])[["rating_P2+", "rating_Heavy"]].mean().rename(
    columns={"rating_P2+": "fc_P2+", "rating_Heavy": "fc_Heavy"}
).reset_index()
fc = fc.merge(hist_mean, on=["network_code", "program_name"], how="left")

# APE & MAPE (skip actual == 0)
fc_p2 = fc[fc["rating_P2+"] > 0].copy()
fc_p2["ape_P2+"] = (fc_p2["rating_P2+"] - fc_p2["fc_P2+"]).abs() / fc_p2["rating_P2+"] * 100.0

fc_hv = fc[fc["rating_Heavy"] > 0].copy()
fc_hv["ape_Heavy"] = (fc_hv["rating_Heavy"] - fc_hv["fc_Heavy"]).abs() / fc_hv["rating_Heavy"] * 100.0

mape_p2 = fc_p2["ape_P2+"].mean()
mape_hv = fc_hv["ape_Heavy"].mean()
top_miss = fc_p2.loc[fc_p2["ape_P2+"].idxmax()]

print("Telecasts forecast (n):", len(fc))
print(f"MAPE, P2+ (%): {mape_p2:.1f}")
print(f"MAPE, heavy buyers (%): {mape_hv:.1f}")
print(f"Biggest miss, P2+ (program, date): {top_miss['program_name']} ({top_miss['network_code']}), {top_miss['bdate']}")

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 38.6
Biggest miss, P2+ (program, date): Krapopolis (FOX), 2024-10-27


## Task 4.2 — Why it missed

In [21]:
misses = fc_p2.nlargest(5, "ape_P2+").copy()
misses["Program"] = misses["program_name"] + " (" + misses["network_code"] + ")"
misses_tbl = misses[["Program", "bdate", "fc_P2+", "rating_P2+", "ape_P2+"]].rename(
    columns={"bdate": "Date", "fc_P2+": "Forecast", "rating_P2+": "Actual", "ape_P2+": "APE (%)"}
)
misses_tbl["Forecast"] = misses_tbl["Forecast"].round(2)
misses_tbl["Actual"] = misses_tbl["Actual"].round(2)
misses_tbl["APE (%)"] = misses_tbl["APE (%)"].round(1)
display(misses_tbl.reset_index(drop=True))

# Inspect schedule on those dates/networks to explain misses
for _, r in misses.iterrows():
    sched = p_big5[(p_big5["bdate"] == r["bdate"]) & (p_big5["network_code"] == r["network_code"])].sort_values("start_ts")
    print(f"\nSchedule {r['bdate']} {r['network_code']}:", list(zip(sched["start_ts"].dt.strftime("%H:%M"), sched["program_name"])))

,Program,Date,Forecast,Actual,APE (%)
0,Krapopolis (FOX),2024-10-27,1.06,0.22,387.8
1,Universal Basic Guys (FOX),2024-10-27,0.68,0.26,162.2
2,Fire Country (CBS),2024-10-27,1.70,0.67,154.8
3,Accused (FOX),2024-10-22,0.95,0.37,154.6
4,What Would You Do? (ABC),2024-10-23,0.44,0.18,138.1



Schedule 2024-10-27 FOX: [('19:15', 'Local / Syndicated Programming'), ('20:00', 'The Simpsons'), ('20:30', 'Universal Basic Guys'), ('21:00', "Bob's Burgers"), ('21:30', 'Krapopolis'), ('22:00', "Bob's Burgers"), ('22:30', 'Krapopolis')]

Schedule 2024-10-27 FOX: [('19:15', 'Local / Syndicated Programming'), ('20:00', 'The Simpsons'), ('20:30', 'Universal Basic Guys'), ('21:00', "Bob's Burgers"), ('21:30', 'Krapopolis'), ('22:00', "Bob's Burgers"), ('22:30', 'Krapopolis')]

Schedule 2024-10-27 CBS: [('19:35', '60 Minutes'), ('20:00', "Poppa's House"), ('20:30', 'Fire Country'), ('20:44', 'Tracker'), ('21:44', 'The Equalizer'), ('22:44', 'Fire Country')]

Schedule 2024-10-22 FOX: [('20:00', 'Murder in a Small Town'), ('21:00', 'Accused'), ('22:00', 'Universal Basic Guys'), ('22:30', 'Murder in a Small Town')]

Schedule 2024-10-23 ABC: [('20:00', 'The Golden Bachelorette'), ('21:30', 'Abbott Elementary'), ('22:00', 'What Would You Do?')]


**Answer:** the five biggest misses were driven by live events and schedule disruptions on those nights—three of the five aired on Sunday, 2024-10-27, when NFL games ran long and pushed the FOX and CBS prime schedules back (FOX opens prime with local programming at 19:15, CBS starts 60 Minutes at 19:35 and its next shows at :44), and Krapopolis (FOX) and Fire Country (CBS) aired twice that night, so one telecast was a late second airing with a much smaller audience; Accused (FOX, 2024-10-22) and What Would You Do? (ABC, 2024-10-23) show no schedule change in programs, so their drop is more likely a repeat episode or stronger competition. The heavy-buyer forecast is noticeably less accurate than the persons-2+ forecast (38.6% vs. 35.8%) because heavy buyers are a much smaller subset of the panel (1,661 panelists vs. all P2+ panelists), adding higher random sampling noise to both the historical baseline and the forecast week's actual ratings. A better forecast would incorporate (1) schedule and competitive context (lead-in program, live sports/specials, and rival-network events) and (2) episode-level and structural features (new vs. repeat episodes, day-of-week/time-slot patterns, or shrinkage toward network/daypart averages for low-sample shows)

## Task 5.1 — The plan

In [22]:
# Build a $1,000,000 plan: 1 high-volume anchor (if <= $650k) + 3 lowest target-CPM shows
budget = 1_000_000
anchor = t2_df.loc[t2_df["Target viewers (K)"].idxmax()]
eff = t2_df[t2_df["Program"] != anchor["Program"]].sort_values("Target CPM ($)").head(3)

spent = 0
plan_rows = []

if anchor["Cost :30 ($)"] <= budget * 0.65:
    c = int(anchor["Cost :30 ($)"])
    spent += c
    plan_rows.append((anchor, 1))

rem = budget - spent
per_show = rem // len(eff)
counts = {}
for _, row in eff.iterrows():
    n = int(per_show // row["Cost :30 ($)"])
    counts[row["Program"]] = n
    spent += n * int(row["Cost :30 ($)"])

# Spend remaining dollars on lowest target-CPM shows that fit
for _, row in eff.iterrows():
    c = int(row["Cost :30 ($)"])
    extra = (budget - spent) // c
    if extra > 0:
        counts[row["Program"]] += int(extra)
        spent += int(extra) * c

for _, row in eff.iterrows():
    if counts[row["Program"]] > 0:
        plan_rows.append((row, counts[row["Program"]]))

out = []
for row, n in plan_rows:
    t_imp_m = n * row["Target viewers (K)"] / 1000.0
    a_imp_m = t_imp_m / row["Composition"]
    out.append({
        "Program": row["Program"],
        "Spots": n,
        "Cost ($)": n * int(row["Cost :30 ($)"]),
        "Target impressions (M)": round(t_imp_m, 2),
        "_adults_imp_m": a_imp_m,
    })

plan_df = pd.DataFrame(out)
total_cost = int(plan_df["Cost ($)"].sum())
total_target_imp = plan_df["Target impressions (M)"].sum()
plan_comp = total_target_imp / plan_df["_adults_imp_m"].sum()
plan_cpm = total_cost / (total_target_imp * 1000.0)

display(plan_df[["Program", "Spots", "Cost ($)", "Target impressions (M)"]])
print("Plan cost ($):", total_cost)
print(f"Plan target impressions (M): {total_target_imp:.1f}")
print(f"Plan target CPM ($): {plan_cpm:.2f} | Composition: {plan_comp:.3f} | Non-target spend (%): {(1 - plan_comp) * 100:.1f}")

,Program,Spots,Cost ($),Target impressions (M)
0,Unsellable Houses (HGTV),26,335400,8.25
1,Halloween Wars (FOOD),34,329800,5.50
2,Grotesquerie (FX),44,334048,5.46


Plan cost ($): 999248
Plan target impressions (M): 19.2
Plan target CPM ($): 52.02 | Composition: 0.223 | Non-target spend (%): 77.7


## Task 5.2 — Recommendation

**Answer:** for Kestrel's \$1,000,000 November flight, we recommend allocating \$999,248 across Unsellable Houses (HGTV, 26 spots), Halloween Wars (FOOD, 34 spots) and Grotesquerie (FX, 44 spots), delivering about 19.2M heavy-buyer impressions at a blended target CPM of about \$52. This mix prioritizes low target-CPM, high-index cable programs to maximize cost efficiency while keeping spend diversified across three cable networks rather than overpaying for a single high-cost slot. Because our October forecast error for heavy buyers was 38.6\% MAPE (driven by schedule swings and small in-target panel sample sizes), actual delivery carries meaningful week-to-week variance. To manage that risk, we should negotiate audience guarantees with make-good provisions so any under-delivery is compensated with additional spots. Based on the plan's blended composition of about 22\%, roughly 78\% of media dollars (about \$780,000) will reach adults who are not heavy athletic-shoe buyers - an expected trade-off in linear TV that still delivers secondary brand awareness among lighter buyers. After the flight airs, we should run a post-buy panel evaluation measuring actual heavy-buyer impressions, GRPs, reach, and frequency against the guarantee to claim any owed make-goods. Finally, pairing post-buy delivery with web traffic or retail sales lift during and after the week will verify whether heavy-buyer reach translated into Kestrel demand